# Uncertainty study (thesis objectives 1 and 2) on Google Colab

A quick version of the full study: **every model (12) on every dataset (11)**, with fewer questions per
dataset (`PER_SET`, default 20). It runs the same code as the Mac run (`python -m uncertainty run-all`).

**Before you start:** *Runtime → Change runtime type → GPU* (T4 is fine; L4 or A100 is faster).

**How to use:** run the cells from top to bottom. The study cell (step 6) runs for hours.

**If Colab disconnects:** reconnect and run every cell again from the top. Results are saved on your Google Drive,
so finished work is kept and the run continues where it stopped. Models are downloaded again (about 10 minutes).

| `PER_SET` | Questions per model | Free T4 | L4 | A100 |
|---|---|---|---|---|
| 20 (default) | 220 | ~10 h | ~5–6 h | ~3 h |
| 30 | 330 | ~15 h | ~8 h | ~4–5 h |

These times are rough. After about 15 minutes the pilot stage measures the real speed, and the projected hours
appear in `run.log` and in `reports/SUMMARY.md` (column *Projected h*).

## 1. Settings

In [ ]:
PER_SET = 20   # questions per dataset (11 datasets, spread evenly so every kind of question stays)

MODELS = ["qwen2.5:3b", "qwen2.5:7b", "qwen2.5:14b", "qwen3:4b-instruct-2507-q4_K_M",
          "llama3.2:3b", "llama3.1:8b", "llama2:7b",
          "gemma3:4b", "gemma3:12b",
          "phi4-mini", "phi4",
          "mistral:7b"]

REPO_URL = "https://github.com/Hemant-Shashikant-Yadav/LLM-Hallucination.git"
BRANCH = "main"
RESULTS_DIR = "/content/drive/MyDrive/LLM-Hallucination/uq_results"   # on Drive, so it survives disconnects
REPO = "/content/LLM-Hallucination"

## 2. Check the GPU and the disk (the 12 models need about 56 GB)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

## 3. Connect Google Drive (results are written here)

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results folder:", RESULTS_DIR)

## 4. Get the code and install the few packages Colab does not already have

In [ ]:
if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q loguru pydantic-settings ollama httpx sentencepiece

## 5. Install and start Ollama (one model loaded at a time, 4 requests in parallel)

In [ ]:
import subprocess, time, urllib.request


def ollama_up():
    try:
        return urllib.request.urlopen("http://localhost:11434/api/version", timeout=2).read().decode()
    except Exception:
        return None


if not ollama_up():
    !apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    env = dict(os.environ, OLLAMA_NUM_PARALLEL="4", OLLAMA_MAX_LOADED_MODELS="1", OLLAMA_KEEP_ALIVE="30m")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
print("Ollama:", ollama_up() or "NOT RUNNING, see /content/ollama.log")

## 6. Run the study

Steps: check (downloads the models, checks the server's log-probs, runs the unit tests) → pilot (measures the
speed) → the 3 focus models (with the Objective 2 re-asks) → the other 9 models → reports. Reports are rebuilt after
every model, so even a run that stops early leaves complete reports for the finished models.

If it stops with a message, read it, fix what it says, and run this cell again: finished work is kept.

In [ ]:
cmd = (f'python -m uncertainty run-all --results-dir "{RESULTS_DIR}" --models {" ".join(MODELS)} '
       f'--memory-gb 16 --limit {PER_SET} --tier-focus extended --tier-others extended --second-judge-n 50')
print(cmd)
!{cmd}

## 7. Summary

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open(f"{RESULTS_DIR}/reports/SUMMARY.md", encoding="utf-8").read()))

## 8. Download the results (optional: they are already on your Drive)

To bring them into the repository, copy the folder into `uncertainty/results/` on your PC.

In [ ]:
!cd "{RESULTS_DIR}/.." && zip -qr /content/uq_results.zip uq_results && ls -lh /content/uq_results.zip
from google.colab import files

files.download("/content/uq_results.zip")